# 第2回 宿題（発展）: 講座の用語集で学習する

`scripts/make_sft_dataset.py` で作った Q&A データ（`messages` 形式）を FAQ の代わりに学習させ，自分で用意した評価用の質問で学習前後を比べる．CONFIG は `num_train_epochs` と `max_new_tokens` だけを変えた．用語集を検索する仕組みとの比較はノートブックの外で行う．

In [ ]:
import subprocess
try:
    gpu_info = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout
except FileNotFoundError:   # CPU ランタイムには nvidia-smi が無い
    gpu_info = ""
print(gpu_info or "GPU が見つかりません。ランタイム → ランタイムのタイプを変更 → T4 GPU を選んでください。")

In [ ]:
# Colab の T4（Python 3.13、torch 2.11.0+cu130）で動作確認した版。torch は Colab 同梱のものをそのまま使う
PINS = {
    "transformers": "5.18.0",   # Qwen3.5 は transformers 5 系が必要
    "peft": "0.21.1",           # LoRA の実装
    "trl": "1.14.2",            # 学習ループ（SFTTrainer）
    "accelerate": "1.15.0",
    "datasets": "5.1.0",
}
spec = " ".join(f'"{k}=={v}"' for k, v in PINS.items())
%pip uninstall -y -q torchao   # Colab 同梱の torchao（0.10）は peft 0.21 と非互換で、学習時に ImportError になる。使わないので外す
%pip install -q {spec}

# cudf-cu13 と pyarrow の依存警告が ERROR 表示されるが、cudf は使わないので無視してよい
import transformers, peft, trl, accelerate, datasets, torch
print({m.__name__: m.__version__ for m in [transformers, peft, trl, accelerate, datasets, torch]})

In [ ]:
from pathlib import Path
import os, json, time

CONFIG = {
    "base_model": "Qwen/Qwen3.5-0.8B",   # 小さな指示チューニング済みモデル（Apache-2.0）。量子化なしで T4 に載る大きさ
    "revision": "2fc06364715b967f1860aea9cf38778875588b17",  # いつ実行しても同じ重みを使うために固定
    "seed": 42,
    "num_train_epochs": 10,  # 第2回 宿題（発展）: 116 件 × 10 周。1 用語に質問が 2 つあるので，各定義は 20 回学習する（演習の各例の回数に合わせた）
    "warmup_steps": 6,
    "lr": 2e-4,
    "lora_r": 16,            # LoRA の行列の次元（小さいほど学習するパラメータが少ない）
    "lora_alpha": 32,
    "max_seq_len": 512,
    "max_new_tokens": 160,  # 定義文は最長 212 字なので演習より長くする
    "gen": {"do_sample": False},   # 評価は貪欲生成で固定（前後比較の条件を揃える）
    "work_dir": "/content/qlora_lab",                          # 結果ファイルの保存先（左のファイル一覧から見られる）
    "drive_dir": "/content/drive/MyDrive/aie2026/qlora_lab",  # USE_DRIVE=True のときの保存先
}
WORK = Path(CONFIG["work_dir"]); WORK.mkdir(parents=True, exist_ok=True)

USE_DRIVE = False  # True にすると Google Drive をマウントして adapter と出力を保存する
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    Path(CONFIG["drive_dir"]).mkdir(parents=True, exist_ok=True)
print(json.dumps(CONFIG, ensure_ascii=False, indent=2))

## 用語集の Q&A データを作る

課題文では Codespaces で作ってアップロードするが，同じスクリプトをここで実行する．

In [ ]:
import subprocess
subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", "lec02", "https://github.com/kyuringo0312/lecture-ai-engineering-2026", "/content/repo"], check=True)
out = subprocess.run(["python", "scripts/make_sft_dataset.py"], cwd="/content/repo", capture_output=True, text=True, check=True).stdout
(WORK / "glossary_sft.jsonl").write_text(out, encoding="utf-8")
records = [json.loads(line) for line in out.splitlines() if line]
print(len(records), "件 | 例:", json.dumps(records[0], ensure_ascii=False)[:200])


In [ ]:
import re
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM
pd.set_option("display.max_colwidth", None)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
tok = AutoTokenizer.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"])
if tok.pad_token is None: tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"], dtype=torch.float32).to(DEVICE)
model.config.use_cache = True
BASE_MEMORY_GB = round(model.get_memory_footprint() / 1e9, 2)
print("loaded:", CONFIG["base_model"], "| device:", DEVICE, "| 重みのメモリ(GB):", BASE_MEMORY_GB)

def generate(model, example):
    # 学習データと同じく system なしの 1 問 1 答．enable_thinking=False は演習と同じ
    prompt = tok.apply_chat_template([{"role": "user", "content": example["question"]}], tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=CONFIG["max_new_tokens"], pad_token_id=tok.pad_token_id, **CONFIG["gen"])
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()


In [ ]:
# 評価用の質問と正解を自分で用意した（must_include: 答えに含まれるべき語句．unknown は用語集に無い語）
EVAL = [
    # known: 学習データと同じ言い回しの質問
    ("known", "「LoRA」とは何ですか？", ["低ランク"]),
    ("known", "「QLoRA」とは何ですか？", ["量子化"]),
    ("known", "「宿題の3レベル」とは何ですか？", ["基礎", "実装", "発展"]),
    ("known", "「answer.md」とは何ですか？", ["lectures/NN/answer.md"]),
    ("known", "「lecNN-start タグ」とは何ですか？", ["lec03-start"]),
    ("known", "「既定モデル」とは何ですか？", ["gemini-3.5-flash-lite"]),
    ("known", "「TestModel」とは何ですか？", ["偽モデル"]),
    ("known", "「Codespaces secret」とは何ですか？", ["環境変数"]),
    ("known", "「過学習」とは何ですか？", ["学習データ"]),
    ("known", "「ハーネス」とは何ですか？", ["ループ"]),
    # paraphrase: 語は含むが，聞き方を変えた質問
    ("paraphrase", "LoRA はどんな手法か簡単に教えてください", ["低ランク"]),
    ("paraphrase", "宿題の3レベルにはどんな段階がありますか", ["基礎", "実装", "発展"]),
    ("paraphrase", "この講座で標準に使う既定モデルは何ですか", ["gemini-3.5-flash-lite"]),
    ("paraphrase", "answer.md はどこにあるファイルですか", ["lectures/NN/answer.md"]),
    ("paraphrase", "過学習に気づくにはどうすればいいですか", ["検証データ"]),
    # reverse: 語を含まず，説明から語を答えさせる質問
    ("reverse", "事前学習済みの重みを固定して小さな行列だけを学習する手法は何と呼びますか", ["LoRA"]),
    ("reverse", "重みを少ないビット数で表してメモリを減らす技術は何と呼びますか", ["量子化"]),
    ("reverse", "LLM が事実と異なる内容をもっともらしく生成する現象は何と呼びますか", ["ハルシネーション"]),
    ("reverse", "1 回の更新でパラメータをどれだけ動かすかを決める値は何と呼びますか", ["学習率"]),
    ("reverse", "この講座で演習の開始時点のコードに付けている Git タグは何と呼びますか", ["lecNN-start"]),
    # unknown: 用語集に無い語（実在 3，架空 3）．知らないと答えるのが望ましいが，学習データに「知らない」の例は無い
    ("unknown_real", "「バッチ正規化」とは何ですか？", []),
    ("unknown_real", "「Docker」とは何ですか？", []),
    ("unknown_real", "「Transformer」とは何ですか？", []),
    ("unknown_fake", "「ゼファーネスト法」とは何ですか？", []),
    ("unknown_fake", "「lecNN-end タグ」とは何ですか？", []),
    ("unknown_fake", "「宿題の5レベル」とは何ですか？", []),
]
REFUSE = re.compile(r"分かりません|わかりません|存じ|ありません|知りません|確認が必要|情報がありません|見つかりません")

def evaluate_glossary(model, label):
    rows, t0 = [], time.time()
    for group, q, must in EVAL:
        raw = generate(model, {"question": q})
        hit = all(w in raw for w in must) if must else None
        rows.append({"group": group, "question": q, "must_include": must, "raw": raw, "content_ok": hit, "refused": bool(REFUSE.search(raw))})
    summary = {"label": label, "seconds": round(time.time() - t0, 1)}
    for g in ["known", "paraphrase", "reverse"]:
        rs = [r for r in rows if r["group"] == g]
        summary[g] = f'{sum(r["content_ok"] for r in rs)}/{len(rs)}'
    for g in ["unknown_real", "unknown_fake"]:
        rs = [r for r in rows if r["group"] == g]
        summary[g + "_refused"] = f'{sum(r["refused"] for r in rs)}/{len(rs)}'
    (WORK / f"eval_{label}.json").write_text(json.dumps({"summary": summary, "rows": rows}, ensure_ascii=False, indent=2), encoding="utf-8")
    print(json.dumps(summary, ensure_ascii=False))
    return summary, rows

before_summary, before_rows = evaluate_glossary(model, "before")
display(pd.DataFrame(before_rows)[["group", "question", "raw", "content_ok", "refused"]])


In [ ]:
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

# make_sft_dataset.py の各行はそのまま messages 形式なので，Dataset.from_list に渡すだけで学習できる
train_ds = Dataset.from_list(records)
model.config.use_cache = False
lora = LoraConfig(r=CONFIG["lora_r"], lora_alpha=CONFIG["lora_alpha"], lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
sft = SFTConfig(output_dir=str(WORK / "ckpt"), num_train_epochs=CONFIG["num_train_epochs"], per_device_train_batch_size=4, gradient_accumulation_steps=2,
                learning_rate=CONFIG["lr"], lr_scheduler_type="cosine", warmup_steps=CONFIG["warmup_steps"], logging_steps=10, save_steps=50, save_total_limit=2,
                max_length=CONFIG["max_seq_len"], packing=False, report_to="none", seed=CONFIG["seed"])
trainer = SFTTrainer(model=model, args=sft, train_dataset=train_ds, peft_config=lora, processing_class=tok)
trainer.model.print_trainable_parameters()
t0 = time.time()
trainer.train()
print(f"学習時間: {time.time()-t0:.0f} 秒 | ステップ数: {trainer.state.global_step} | エポック: {trainer.state.epoch:.1f}")
trainer.model.save_pretrained(WORK / "adapter")
tuned = trainer.model
tuned.config.use_cache = True
_ = tuned.eval()


In [ ]:
after_summary, after_rows = evaluate_glossary(tuned, "after")
display(pd.DataFrame([before_summary, after_summary]).set_index("label"))
display(pd.DataFrame([{"group": b["group"], "質問": b["question"], "学習前": b["raw"], "学習後": a["raw"], "前": b["content_ok"], "後": a["content_ok"], "後 refused": a["refused"]} for b, a in zip(before_rows, after_rows)]))
import transformers, peft, trl
result = {"config": {k: v for k, v in CONFIG.items() if k not in ("work_dir", "drive_dir")},
          "versions": {m.__name__: m.__version__ for m in [transformers, peft, trl, torch]},
          "method": "LoRA（量子化なし）・用語集 Q&A で学習", "train_size": len(records),
          "before": before_summary, "after": after_summary, "executed_training_myself": True}
(WORK / "result.json").write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(result, ensure_ascii=False, indent=2))


In [ ]:
# 結果を 1 つの zip にまとめてダウンロードする（第2回の提出用に追加したセル）
import zipfile
RUN_NAME = "adv_glossary"
log = trainer.state.log_history
trainable, total = trainer.model.get_nb_trainable_parameters()
adapter_bytes = sum(p.stat().st_size for p in (WORK / "adapter").rglob("*") if p.is_file())
extra = {"run": RUN_NAME, "trainable_params": trainable, "all_params": total, "trainable_percent": round(100 * trainable / total, 3),
         "adapter_mb": round(adapter_bytes / 1e6, 2), "base_memory_gb": BASE_MEMORY_GB, "log_history": log}
(WORK / "train_log.json").write_text(json.dumps(extra, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps({k: v for k, v in extra.items() if k != "log_history"}, ensure_ascii=False))
print("losses:", [(h["step"], round(h["loss"], 4)) for h in log if "loss" in h])
print("train:", [h for h in log if "train_runtime" in h])
zip_path = f"/content/lec02_{RUN_NAME}.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for p in WORK.glob("*.json*"):
        z.write(p, p.name)
print("saved:", zip_path, [p.name for p in WORK.glob("*.json*")])
from google.colab import files
files.download(zip_path)
